In [1]:
import pandas as pd
import time
from pytrends.request import TrendReq
from datetime import datetime, timedelta
import os

pytrends = TrendReq(hl='fr-FR', tz=60, timeout=(10, 25))

GEO = "FR"
OUTPUT_DIR = "/Users/r/Documents/CS/projet_S8/data/google_trends"
os.makedirs(OUTPUT_DIR, exist_ok=True)

KEYWORDS = [
    "Journée nationale des aidants",
    "Journée mondiale de lutte contre le cancer",
    "Journée mondiale de lutte contre l'obésité",
    "Journée mondiale de la trisomie 21",
    "World Health Day",
    "Journée mondiale sans tabac",
    "Journée mondiale du handicap",
    "Journée mondiale de la santé mentale",
]

def keyword_to_filename(kw: str) -> str:
    clean = kw.lower()
    clean = clean.replace(" ", "_").replace("'", "").replace("é", "e").replace("è", "e").replace("ê", "e")
    clean = clean.replace("â", "a").replace("î", "i").replace("ô", "o").replace("û", "u")
    clean = clean.replace("ï", "i").replace("ù", "u").replace("ç", "c")
    return f"{clean}_daily_2016_2026_FR.csv"

def generate_periods(start: str, end: str, days: int = 90):
    periods = []
    current = datetime.strptime(start, "%Y-%m-%d")
    end_dt = datetime.strptime(end, "%Y-%m-%d")
    while current < end_dt:
        next_dt = current + timedelta(days=days)
        if next_dt > end_dt:
            next_dt = end_dt
        periods.append((current.strftime("%Y-%m-%d"), next_dt.strftime("%Y-%m-%d")))
        current = next_dt
    return periods

def fetch_chunk(keyword: str, start: str, end: str, retries: int = 5) -> pd.DataFrame:
    timeframe = f"{start} {end}"
    for attempt in range(retries):
        try:
            pytrends.build_payload([keyword], cat=0, timeframe=timeframe, geo=GEO, gprop='')
            df = pytrends.interest_over_time()
            if df is not None and not df.empty:
                return df[[keyword]].copy()
            print(f"  ⚠️  Vide pour {timeframe} → rempli avec 0")
            date_range = pd.date_range(start=start, end=end, freq='D')
            return pd.DataFrame({keyword: 0}, index=date_range)
        except Exception as e:
            wait = 30 * (attempt + 1)
            print(f"  ❌ Erreur (tentative {attempt+1}/{retries}) : {e} — attente {wait}s")
            time.sleep(wait)
    print(f"  🚫 Échec total pour {timeframe} → rempli avec 0")
    date_range = pd.date_range(start=start, end=end, freq='D')
    return pd.DataFrame({keyword: 0}, index=date_range)

def fetch_keyword(keyword: str):
    output_file = os.path.join(OUTPUT_DIR, keyword_to_filename(keyword))
    periods = generate_periods("2016-01-01", "2026-01-01", days=90)
    print(f"\n{'='*60}")
    print(f"🔍 Keyword : {keyword}")
    print(f"📅 {len(periods)} chunks à récupérer...")
    print(f"{'='*60}")

    chunks = []
    for i, (start, end) in enumerate(periods):
        print(f"  [{i+1}/{len(periods)}] Fetch : {start} {end}")
        df = fetch_chunk(keyword, start, end)
        chunks.append(df)
        time.sleep(5)

    final_df = pd.concat(chunks)
    final_df = final_df[~final_df.index.duplicated(keep='first')]
    final_df = final_df.sort_index()
    final_df.index.name = "date"
    final_df.columns = ["interet_relatif"]
    final_df.to_csv(output_file)

    print(f"\n  ✅ Export : {output_file}")
    print(f"     Lignes totales    : {len(final_df)}")
    print(f"     Lignes non-nulles : {(final_df['interet_relatif'] > 0).sum()}")
    print(f"     Période           : {final_df.index.min().date()} → {final_df.index.max().date()}")

# MAIN
print(f"🚀 Lancement pour {len(KEYWORDS)} keywords\n")
for kw in KEYWORDS:
    fetch_keyword(kw)
    print(f"\n⏳ Pause 15s avant le prochain keyword...")
    time.sleep(15)

print(f"\n\n🎉 Terminé ! Tous les CSV sont dans {OUTPUT_DIR}")

🚀 Lancement pour 8 keywords


🔍 Keyword : Journée nationale des aidants
📅 41 chunks à récupérer...
  [1/41] Fetch : 2016-01-01 2016-03-31
  [2/41] Fetch : 2016-03-31 2016-06-29
  ⚠️  Vide pour 2016-03-31 2016-06-29 → rempli avec 0
  [3/41] Fetch : 2016-06-29 2016-09-27
  ⚠️  Vide pour 2016-06-29 2016-09-27 → rempli avec 0
  [4/41] Fetch : 2016-09-27 2016-12-26
  ⚠️  Vide pour 2016-09-27 2016-12-26 → rempli avec 0
  [5/41] Fetch : 2016-12-26 2017-03-26
  ⚠️  Vide pour 2016-12-26 2017-03-26 → rempli avec 0
  [6/41] Fetch : 2017-03-26 2017-06-24
  ⚠️  Vide pour 2017-03-26 2017-06-24 → rempli avec 0
  [7/41] Fetch : 2017-06-24 2017-09-22
  ⚠️  Vide pour 2017-06-24 2017-09-22 → rempli avec 0
  [8/41] Fetch : 2017-09-22 2017-12-21
  ⚠️  Vide pour 2017-09-22 2017-12-21 → rempli avec 0
  [9/41] Fetch : 2017-12-21 2018-03-21
  ⚠️  Vide pour 2017-12-21 2018-03-21 → rempli avec 0
  [10/41] Fetch : 2018-03-21 2018-06-19
  ⚠️  Vide pour 2018-03-21 2018-06-19 → rempli avec 0
  [11/41] Fetch : 2018-0